# Degree Zero — QE with COMETKiwi

Scores each condition against the source with **reference-free quality estimation** (`Unbabel/wmt22-cometkiwi-da`, 0–1, higher = better).

**Why an isolated environment:** `unbabel-comet` pins `numpy<2`, `huggingface-hub<1.0` and old `torchmetrics`. Installed into Colab's own Python, pip silently falls back to a 2023 COMET and downgrades `transformers` / `huggingface-hub`, breaking everything else. Here COMET lives in its own Python 3.11 venv and is called as a script; the analysis runs in Colab's normal Python.

**One-time setup**
1. On huggingface.co, open `Unbabel/wmt22-cometkiwi-da` and accept the license (auto-approved).
2. Create a read token (Settings → Access Tokens); add it in Colab's 🔑 **Secrets** panel as `HF_TOKEN` with notebook access on.
3. **Fresh** T4 runtime (if you already ran `pip install unbabel-comet`: Runtime → Disconnect and delete runtime).

**Caveat:** COMETKiwi also rewards fluency, so G (an ungrammatical gloss) scores low for fluency reasons — compare L / F / F2 / A with each other, not with G.

In [ ]:
# isolated COMET environment (~2–3 min, mostly the torch download)
!pip install -q uv
!uv venv /content/qe_env --python 3.11 -q
!uv pip install -q --python /content/qe_env/bin/python unbabel-comet pandas "setuptools<81"   # old torchmetrics needs pkg_resources
!/content/qe_env/bin/python -c "import importlib.metadata as m, numpy; print('unbabel-comet', m.version('unbabel-comet'), '| numpy', numpy.__version__)"

In [ ]:
import os, numpy as np, pandas as pd
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # inherited by the scoring subprocess

if not os.path.exists("/content/results.csv"):
    from google.colab import files
    files.upload()   # pick results.csv from the pilot notebook
print(pd.read_csv("/content/results.csv").shape)

In [ ]:
SCRIPT = """
import os, sys, pandas as pd
from huggingface_hub import login
from comet import download_model, load_from_checkpoint

login(token=os.environ["HF_TOKEN"])
df = pd.read_csv(sys.argv[1])
model = load_from_checkpoint(download_model("Unbabel/wmt22-cometkiwi-da"))
for c in ["G", "L", "F", "F2", "A"]:
    if c not in df:
        continue
    data = [{"src": s, "mt": str(m)} for s, m in zip(df.S, df[c])]
    df[f"qe_{c}"] = model.predict(data, batch_size=16, gpus=1, progress_bar=False).scores
    print("scored", c, flush=True)
df.to_csv(sys.argv[2], index=False)
"""
open("/content/score_qe.py", "w").write(SCRIPT)

In [ ]:
!/content/qe_env/bin/python /content/score_qe.py /content/results.csv /content/results_qe.csv

## Results

In [ ]:
df = pd.read_csv("/content/results_qe.csv")
ok = df[~df.bad] if "bad" in df else df
CONDS = [c for c in ["G", "L", "F", "F2", "A"] if f"qe_{c}" in ok]
display(pd.DataFrame({c: {"QE": ok[f"qe_{c}"].median(),
                          "int_G": ok[f"int_G_{c}"].median() if f"int_G_{c}" in ok else np.nan}
                      for c in CONDS}).T.round(3))

## RQ2 — does intervention cost meaning?
- **Noise:** QE(F) vs QE(F2) — how much QE moves under mere rewording.
- **Within item, L → F:** F intervenes more than L; does QE drop? And do items where F moved further from G (relative to L) lose more QE? This uses the per-item score int_G(F) − int_G(L), which removes the source-paragraph effect.
- **Proxy check:** do round-trip drift and LaBSE xsim agree with COMETKiwi?

In [ ]:
from scipy.stats import wilcoxon, spearmanr
def paired(a, b, label):
    diff = ok[a] - ok[b]
    print(f"{label:28s} median diff={diff.median():+.4f}  {(diff > 0).mean():.0%} of items  "
          f"p={wilcoxon(ok[a], ok[b]).pvalue:.3g}")

paired("qe_F", "qe_F2", "QE(F) − QE(F2)  [noise]")
paired("qe_L", "qe_F", "QE(L) − QE(F)")
if "qe_A" in ok:
    paired("qe_F", "qe_A", "QE(F) − QE(A)")
print()

def rho(x, y, label):
    r = spearmanr(x, y)
    print(f"{label:50s} rho = {r.statistic:+.3f}  p = {r.pvalue:.3g}")

rho(ok.int_G_F - ok.int_G_L, ok.qe_F - ok.qe_L, "per item: Δint_G(F−L) vs ΔQE(F−L)")
if "qe_A" in ok:
    rho(ok.int_G_A - ok.int_G_F, ok.qe_A - ok.qe_F, "per item: Δint_G(A−F) vs ΔQE(A−F)")
rho(ok.int_G_F, ok.qe_F, "raw int_G(F) vs QE(F)  [confounded by source]")
print()
for proxy in ["drift_F", "xsim_F"]:
    if proxy in ok:
        rho(ok[proxy], ok.qe_F, f"proxy check: {proxy} vs QE(F)")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for i, c in enumerate([c for c in ["L", "F", "F2", "A"] if c in CONDS]):
    axes[0].scatter(ok[f"int_G_{c}"], ok[f"qe_{c}"], label=c, s=22, c=f"C{i}")
axes[0].set_xlabel("int_G (distance from gloss)"); axes[0].set_ylabel("COMETKiwi QE")
axes[0].set_title("Raw: all conditions"); axes[0].legend()

dx, dy = ok.int_G_F - ok.int_G_L, ok.qe_F - ok.qe_L
axes[1].scatter(dx, dy, s=22)
axes[1].axhline(0, c="gray", ls="--"); axes[1].axvline(0, c="gray", ls="--")
axes[1].set_xlabel("Δ intervention  int_G(F) − int_G(L)"); axes[1].set_ylabel("Δ QE  QE(F) − QE(L)")
axes[1].set_title("Per item: does extra intervention cost quality?")
plt.tight_layout(); plt.show()

xm, ym = dx.median(), dy.median()
q = np.select([(dx <= xm) & (dy >= ym), (dx > xm) & (dy >= ym), (dx > xm) & (dy < ym)],
              ["literal-faithful", "good paraphrase", "lossy paraphrase"], "error? inspect")
print(pd.Series(q).value_counts())

In [ ]:
from google.colab import files
files.download("/content/results_qe.csv")